<a href="https://colab.research.google.com/github/bohdanakhobta/ab-test-conversion-analysis/blob/main/ab_test_conversion_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required library to connect BigQuery
!pip install --upgrade google-cloud-bigquery -q

from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

# Authenticate and connect to BigQuery
auth.authenticate_user()

project_id = "data-analytics-mate"
client = bigquery.Client(project=project_id)

In [ ]:
query = """
with session_info as (

  select
        s.date,
        s.ga_session_id,
        sp.country,
        sp.device,
        sp.continent,
        sp.channel,
        ab.test,
        ab.test_group
  from `DA.ab_test` ab
  join `DA.session` s
  on ab.ga_session_id = s.ga_session_id
  join `DA.session_params`sp
  on sp.ga_session_id = ab.ga_session_id

),


session_with_orders as (
SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count(distinct o.ga_session_id) as session_with_orders
from `DA.order` o
join session_info
on o.ga_session_id = session_info.ga_session_id
group by
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group

),
events as (
select
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        sp.event_name,
        count(sp.ga_session_id) as event_cnt
from `DA.event_params` sp
join session_info
on sp.ga_session_id = session_info.ga_session_id
group by
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        sp.event_name
),
session as (
SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count(distinct session_info.ga_session_id) as session_cnt
from session_info
group by
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
),
account as (
select
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count(distinct acs.ga_session_id) as new_account_cnt
from `DA.account_session` acs
join session_info
on acs.ga_session_id = session_info.ga_session_id
group by
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group

)



SELECT
        session_with_orders.date,
        session_with_orders.country,
        session_with_orders.device,
        session_with_orders.continent,
        session_with_orders.channel,
        session_with_orders.test,
        session_with_orders.test_group,
        'session with orders' as event_name,
        session_with_orders.session_with_orders as value
from session_with_orders

union all

SELECT
        events.date,
        events.country,
        events.device,
        events.continent,
        events.channel,
        events.test,
        events.test_group,
        event_name,
        event_cnt as value
from events

union all

SELECT
        session.date,
        session.country,
        session.device,
        session.continent,
        session.channel,
        session.test,
        session.test_group,
        'session' as event_name,
        session_cnt as value
from session

union all

SELECT
        account.date,
        account.country,
        account.device,
        account.continent,
        account.channel,
        account.test,
        account.test_group,
        'new account' as event_name,
        new_account_cnt as value
from account

"""
df = client.query(query).to_dataframe()
df.head() # Dataframe output

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-12-08,Palestine,desktop,Asia,Direct,4,2,new account,1
1,2020-12-08,Palestine,desktop,Asia,Direct,3,2,new account,1
2,2020-11-06,Puerto Rico,desktop,Americas,Social Search,2,2,new account,1
3,2020-11-06,Puerto Rico,desktop,Americas,Social Search,1,1,new account,1
4,2020-12-08,Croatia,desktop,Europe,Direct,4,2,new account,1


In [ ]:
# Aggregate data by test and test_group and event_name -> prepare data for metric calculation
agg = (
    df
    .groupby(['test', 'test_group', 'event_name'], as_index=False)
    .agg({'value': 'sum'})
)

agg.head()

,test,test_group,event_name,value
0,1,1,add_payment_info,1988
1,1,1,add_shipping_info,3034
2,1,1,add_to_cart,1395
3,1,1,begin_checkout,3784
4,1,1,click,368


In [ ]:
# Define metrics as numerator / denominator pairs
metrics = {
    'add_payment': ('add_payment_info', 'session'),
    'add_shipping': ('add_shipping_info', 'session'),
    'begin_checkout': ('begin_checkout', 'session'),
    'new_account': ('new account', 'session')
}

In [ ]:
# Function to extract value for specific event
def get_value(data, test_id, group, event):
    value = data[
        (data['test'] == test_id) &
        (data['test_group'] == group) &
        (data['event_name'] == event)
    ]['value'].sum()

    return value

In [ ]:
import numpy as np
from scipy import stats

# Function for two-proportion z-test -> check if difference between groups is statistically significant
def z_test(control_success, control_total, test_success, test_total):

    p1 = control_success / control_total
    p2 = test_success / test_total

    p_pool = (control_success + test_success) / (control_total + test_total)

    z = (p2 - p1) / np.sqrt(p_pool * (1 - p_pool) * (1/control_total + 1/test_total))

    p_value = 2 * (1 - stats.norm.cdf(abs(z)))

    return z, p_value

In [ ]:
groups = sorted(agg['test_group'].unique())
groups

[np.int64(1), np.int64(2)]

In [ ]:
# Define control and test groups
control_group = 1
test_group = 2

results = []

# Loop through all tests and metrics
tests = agg['test'].unique()

for test_id in tests:

    for metric_name, (num_event, denom_event) in metrics.items():

        # Extract control values
        control_num = get_value(agg, test_id, control_group, num_event)
        control_den = get_value(agg, test_id, control_group, denom_event)

        # Extract test values
        test_num = get_value(agg, test_id, test_group, num_event)
        test_den = get_value(agg, test_id, test_group, denom_event)

        # Skip if no data
        if control_den == 0 or test_den == 0:
            continue

        # Conversion rates
        cr_control = control_num / control_den
        cr_test = test_num / test_den

        # Relative change (%)
        metric_change = (cr_test - cr_control) / cr_control * 100

        # Statistical test
        z_stat, p_value = z_test(control_num, control_den, test_num, test_den)

        results.append({
            'test_number': test_id,
            'metric': metric_name,
            'numerator_event': num_event,
            'denominator_event': denom_event,
            'numerator_control': control_num,
            'denominator_control': control_den,
            'conversion_rate_control': cr_control,
            'numerator_test': test_num,
            'denominator_test': test_den,
            'conversion_rate_test': cr_test,
            'metric_change_%': metric_change,
            'z_stat': z_stat,
            'p_value': p_value,
            'significant': p_value < 0.05
        })

final_df = pd.DataFrame(results)
final_df

,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_%,z_stat,p_value,significant
0,1,add_payment,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,12.542021,3.924884,0.000087,True
1,1,add_shipping,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,6.560481,2.603571,0.009226,True
2,1,begin_checkout,begin_checkout,session,3784,45362,0.083418,4021,45193,0.088974,6.660587,2.978783,0.002894,True
3,1,new_account,new account,session,3823,45362,0.084278,3681,45193,0.081451,-3.354299,-1.542883,0.122859,False
4,2,add_payment,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,3.576911,1.240994,0.214608,False
5,2,add_shipping,add_shipping_info,session,3480,50637,0.068724,3510,50244,0.069859,1.650995,0.709557,0.477979,False
6,2,begin_checkout,begin_checkout,session,4262,50637,0.084168,4313,50244,0.085841,1.988164,0.952898,0.340642,False
7,2,new_account,new account,session,4165,50637,0.082252,4184,50244,0.083274,1.241934,0.588793,0.556000,False
8,3,add_payment,add_payment_info,session,3623,70047,0.051722,3697,70439,0.052485,1.474630,0.643172,0.520112,False
9,3,add_shipping,add_shipping_info,session,5298,70047,0.075635,5188,70439,0.073652,-2.621211,-1.413727,0.157442,False


In [ ]:
# Check final structure
final_df.head()

# Check data types and nulls
final_df.info()

# Basic validation of conversion rates
final_df[['conversion_rate_control', 'conversion_rate_test']].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   test_number              16 non-null     int64  
 1   metric                   16 non-null     object 
 2   numerator_event          16 non-null     object 
 3   denominator_event        16 non-null     object 
 4   numerator_control        16 non-null     int64  
 5   denominator_control      16 non-null     int64  
 6   conversion_rate_control  16 non-null     float64
 7   numerator_test           16 non-null     int64  
 8   denominator_test         16 non-null     int64  
 9   conversion_rate_test     16 non-null     float64
 10  metric_change_%          16 non-null     float64
 11  z_stat                   16 non-null     float64
 12  p_value                  16 non-null     float64
 13  significant              16 non-null     bool   
dtypes: bool(1), float64(5), int6

,conversion_rate_control,conversion_rate_test
count,16.000000,16.000000
mean,0.074760,0.074933
std,0.026927,0.025622
min,0.035507,0.034249
25%,0.050992,0.051694
50%,0.078944,0.077552
75%,0.084195,0.083915
max,0.136080,0.131518


In [ ]:
# Aggregate data by device
agg_device = (
    df
    .groupby(['test', 'test_group', 'device', 'event_name'], as_index=False)
    .agg({'value': 'sum'})
)

results_device = []

for test_id in agg_device['test'].unique():
    for device in agg_device['device'].unique():

        for metric_name, (num_event, denom_event) in metrics.items():

            def get_val(group, event):
                return agg_device[
                    (agg_device['test'] == test_id) &
                    (agg_device['test_group'] == group) &
                    (agg_device['device'] == device) &
                    (agg_device['event_name'] == event)
                ]['value'].sum()

            control_num = get_val(control_group, num_event)
            control_den = get_val(control_group, denom_event)
            test_num = get_val(test_group, num_event)
            test_den = get_val(test_group, denom_event)

            if control_den == 0 or test_den == 0:
                continue

            cr_control = control_num / control_den
            cr_test = test_num / test_den

            z_stat, p_value = z_test(control_num, control_den, test_num, test_den)

            results_device.append({
                'test_number': test_id,
                'dimension': 'device',
                'dimension_value': device,
                'metric': metric_name,
                'cr_control': cr_control,
                'cr_test': cr_test,
                'metric_change_%': (cr_test - cr_control) / cr_control * 100,
                'p_value': p_value,
                'significant': p_value < 0.05
            })

df_device = pd.DataFrame(results_device)
df_device.head()

,test_number,dimension,dimension_value,metric,cr_control,cr_test,metric_change_%,p_value,significant
0,1,device,desktop,add_payment,0.042695,0.047545,11.360819,0.007210,True
1,1,device,desktop,add_shipping,0.064647,0.072529,12.193247,0.000336,True
2,1,device,desktop,begin_checkout,0.079646,0.091002,14.257595,0.000003,True
3,1,device,desktop,new_account,0.083236,0.081273,-2.357527,0.411526,False
4,1,device,mobile,add_payment,0.045262,0.053020,17.140683,0.000701,True


In [ ]:
# Small country segments may produce unstable conversion rates.
# To avoid division by zero and unreliable statistical results it's important to apply a minimum sample size before calculating metrics.

MIN_SESSIONS = 500

# Function to calculate relative uplift
def calculate_metric_change(cr_control, cr_test):
    if cr_control == 0:
        return np.nan
    return (cr_test - cr_control) / cr_control * 100


# Aggregate data by country
agg_country = (
    df
    .groupby(['test', 'test_group', 'country', 'event_name'], as_index=False)
    .agg({'value': 'sum'})
)

results_country = []

for test_id in agg_country['test'].unique():
    for country in agg_country['country'].dropna().unique():

        for metric_name, (num_event, denom_event) in metrics.items():

            # Function to extract values for a specific group, country, and event
            def get_val(group, event):
                return agg_country[
                    (agg_country['test'] == test_id) &
                    (agg_country['test_group'] == group) &
                    (agg_country['country'] == country) &
                    (agg_country['event_name'] == event)
                ]['value'].sum()

            # Extract numerator and denominator for control and test groups
            control_num = get_val(control_group, num_event)
            control_den = get_val(control_group, denom_event)
            test_num = get_val(test_group, num_event)
            test_den = get_val(test_group, denom_event)

            # Skip segments with insufficient sample size
            if control_den < MIN_SESSIONS or test_den < MIN_SESSIONS:
                continue

            # Skip cases where control conversion rate is zero
            if control_num == 0:
                continue

            # Calculate conversion rates
            cr_control = control_num / control_den
            cr_test = test_num / test_den

            # Skip invalid cases for statistical test
            if cr_control == 0 or cr_test == 0:
                continue

            # Calculate statistical significance
            z_stat, p_value = z_test(control_num, control_den, test_num, test_den)

            # Save results
            results_country.append({
                'test_number': test_id,
                'dimension': 'country',
                'dimension_value': country,
                'metric': metric_name,
                'numerator_event': num_event,
                'denominator_event': denom_event,
                'numerator_control': control_num,
                'denominator_control': control_den,
                'conversion_rate_control': cr_control,
                'numerator_test': test_num,
                'denominator_test': test_den,
                'conversion_rate_test': cr_test,
                'metric_change_%': calculate_metric_change(cr_control, cr_test),
                'z_stat': z_stat,
                'p_value': p_value,
                'significant': p_value < 0.05
            })

df_country = pd.DataFrame(results_country)

# Sort results by p-value
df_country = df_country.sort_values(by='p_value')

df_country.head()

,test_number,dimension,dimension_value,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_%,z_stat,p_value,significant
306,4,country,Taiwan,begin_checkout,begin_checkout,session,302,1785,0.169188,154,1852,0.083153,-50.851415,-7.832831,4.662937e-15,True
274,4,country,Mexico,begin_checkout,begin_checkout,session,58,889,0.065242,139,881,0.157775,141.831383,6.189087,6.051382e-10,True
206,3,country,United States,begin_checkout,begin_checkout,session,4322,30734,0.140626,3896,30833,0.126358,-10.145985,-5.204813,1.941925e-07,True
234,4,country,France,begin_checkout,begin_checkout,session,296,2079,0.142376,192,2107,0.091125,-35.997127,-5.166195,2.389072e-07,True
305,4,country,Taiwan,add_shipping,add_shipping_info,session,119,1785,0.066667,63,1852,0.034017,-48.974082,-4.514676,6.341367e-06,True


In [ ]:
# Aggregate data by channel
agg_channel = (
    df
    .groupby(['test', 'test_group', 'channel', 'event_name'], as_index=False)
    .agg({'value': 'sum'})
)

results_channel = []

for test_id in agg_channel['test'].unique():
    for channel in agg_channel['channel'].unique():

        for metric_name, (num_event, denom_event) in metrics.items():

            def get_val(group, event):
                return agg_channel[
                    (agg_channel['test'] == test_id) &
                    (agg_channel['test_group'] == group) &
                    (agg_channel['channel'] == channel) &
                    (agg_channel['event_name'] == event)
                ]['value'].sum()

            control_num = get_val(control_group, num_event)
            control_den = get_val(control_group, denom_event)
            test_num = get_val(test_group, num_event)
            test_den = get_val(test_group, denom_event)

            if control_den == 0 or test_den == 0:
                continue

            cr_control = control_num / control_den
            cr_test = test_num / test_den

            z_stat, p_value = z_test(control_num, control_den, test_num, test_den)

            results_channel.append({
                'test_number': test_id,
                'dimension': 'channel',
                'dimension_value': channel,
                'metric': metric_name,
                'cr_control': cr_control,
                'cr_test': cr_test,
                'metric_change_%': (cr_test - cr_control) / cr_control * 100,
                'p_value': p_value,
                'significant': p_value < 0.05
            })

df_channel = pd.DataFrame(results_channel)
df_channel.head()

,test_number,dimension,dimension_value,metric,cr_control,cr_test,metric_change_%,p_value,significant
0,1,channel,Direct,add_payment,0.036666,0.049802,35.825180,0.000003,True
1,1,channel,Direct,add_shipping,0.062108,0.069105,11.265775,0.040295,True
2,1,channel,Direct,begin_checkout,0.076981,0.088312,14.719677,0.002821,True
3,1,channel,Direct,new_account,0.085399,0.082038,-3.935085,0.378860,False
4,1,channel,Organic Search,add_payment,0.040829,0.032883,-19.461427,0.000191,True


In [ ]:
df_all_segments = pd.concat([df_device, df_country, df_channel])
df_all_segments.head()

,test_number,dimension,dimension_value,metric,cr_control,cr_test,metric_change_%,p_value,significant,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,z_stat
0,1,device,desktop,add_payment,0.042695,0.047545,11.360819,0.007210,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,device,desktop,add_shipping,0.064647,0.072529,12.193247,0.000336,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,device,desktop,begin_checkout,0.079646,0.091002,14.257595,0.000003,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,device,desktop,new_account,0.083236,0.081273,-2.357527,0.411526,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,device,mobile,add_payment,0.045262,0.053020,17.140683,0.000701,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The A/B test evaluated the impact of the new variant on four key metrics: add_payment_info, add_shipping_info, begin_checkout, and new_account, each measured as a conversion rate relative to total sessions.

At the overall level, the conversion rates for both control and test groups are very close. On average, conversion rates are around 7-7.5%, with only minor differences between the groups. The relative changes (uplift) are small and do not show a consistent positive trend across all metrics.

From a statistical perspective, the majority of p-values are above 0.05, which means that the observed differences are not statistically significant. In practical terms, this suggests that the changes in conversion rates are likely due to random variation rather than the effect of the new variant.

Segment-level analysis (by device, country, and channel) shows some variability. For example, certain segments (such as specific devices or countries) demonstrate higher or lower conversion rates in the test group. However, these differences are either not statistically significant, or
based on relatively small sample sizes, which reduces their reliability.

For next steps, it would be reasonable to continue the experiment to collect more data, or refine the hypothesis and test a more impactful product change

Additionally, segment-level insights can be used to identify user groups where the variant shows potential and design more targeted experiments in the future.

In [ ]:
final_df.columns

Index(['test_number', 'metric', 'numerator_event', 'denominator_event',
       'numerator_control', 'denominator_control', 'conversion_rate_control',
       'numerator_test', 'denominator_test', 'conversion_rate_test',
       'metric_change_pct', 'z_stat', 'p_value', 'significant'],
      dtype='object')

In [ ]:
tableau_df = final_df.copy()

# Select needed columns
tableau_df = tableau_df[
    [
        'test_number',
        'metric',
        'conversion_rate_control',
        'conversion_rate_test',
        'metric_change_pct',
        'p_value',
        'z_stat',
        'significant'
    ]
]

# Add readable label for Tableau
tableau_df['significance_status'] = tableau_df['significant'].apply(
    lambda x: 'Significant' if x else 'Not Significant'
)

# Save to Excel
tableau_df.to_excel('ab_test_results_final.xlsx', index=False)

## Impact of A/B Test on Key Conversion Metrics

https://public.tableau.com/app/profile/bohdana.khobta/viz/ImpactofABTestonKeyConversionMetrics/ABtest_metrics?publish=yes



In [ ]:
final_df[['metric', 'p_value']]

,metric,p_value
0,add_payment,0.000087
1,add_shipping,0.009226
2,begin_checkout,0.002894
3,new_account,0.122859
4,add_payment,0.214608
5,add_shipping,0.477979
6,begin_checkout,0.340642
7,new_account,0.556000
8,add_payment,0.520112
9,add_shipping,0.157442
